## load and prepare data

In [ ]:
%cd ../..
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.insert(0, 'evaluation_meldgraph')
from vol_eval_plots import load_and_prepare_data


In [ ]:
eval_stats_df = load_and_prepare_data()

## cohort overview

### Figure 1

In [ ]:
# one marker per participant, plot participants in a grid, with marker shape for group and color for site
df_overview = eval_stats_df.drop_duplicates(subset='site_subj_id').copy()

# order: by site, then by group (patient, control, disease control) within each site
group_order = ['patient', 'control', 'disease_control']
df_overview['group'] = pd.Categorical(df_overview['group'], categories=group_order, ordered=True)
df_overview = df_overview.sort_values(['site', 'group', 'site_subj_id']).reset_index(drop=True)

# marker shapes per group, with display labels for the legend; only the groups this cohort
# actually has - the FCD selection above leaves no disease controls
group_markers = {'patient': 's', 'control': 'o', 'disease_control': 'h'}
group_labels = {'patient': 'FCD', 'control': 'Control', 'disease_control': 'Disease control'}
group_markers = {group: marker for group, marker in group_markers.items()
                 if (df_overview['group'] == group).any()}

# arrange symbols in a grid
n_per_row = 20
df_overview['col'] = df_overview.index % n_per_row
df_overview['row'] = df_overview.index // n_per_row
# flip row so that the first participant appears at the top
df_overview['row'] = df_overview['row'].max() - df_overview['row']

# assign a color per site
sites = sorted(df_overview['site'].unique())
palette = sns.color_palette('pastel', n_colors=len(sites))
site_colors = dict(zip(sites, palette))

fig, ax = plt.subplots(figsize=(n_per_row * 0.3 + 0.5, (df_overview['row'].max() + 1) * 0.5))

for group, marker in group_markers.items():
    data_group = df_overview[df_overview['group'] == group]
    ax.scatter(
        data_group['col'],
        data_group['row'],
        marker=marker,
        c=[site_colors[s] for s in data_group['site']],
        s=200,
        edgecolor='black',
        linewidth=0.3,
        label=group_labels[group],
    )

ax.set_xlim(-1, n_per_row)
ax.set_ylim(-1, df_overview['row'].max() + 1)
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect('equal')

# legend for group (marker shape)
group_handles = [plt.Line2D([0], [0], marker=marker, color='w', markerfacecolor='gray',
                            markeredgecolor='black', markersize=12, label=group_labels[group])
                 for group, marker in group_markers.items()]
fig.legend(handles=group_handles,
           title='',
           loc='upper center',
           bbox_to_anchor=(.75, 0.2),
           frameon=False,
           fontsize=12,
           title_fontsize=9,
           ncols=1)

# legend for site (color)
site_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=site_colors[s],
                           markeredgecolor='black', markersize=12, label=s)
                for s in sites]
fig.legend(handles=site_handles,
           title='',
           loc='upper center',
           bbox_to_anchor=(.32, 0.2),
           frameon=False,
           fontsize=12,
           title_fontsize=9,
           ncols=2)

print(f"{len(df_overview)} participants")
plt.show()